# LLM & Agent Evaluation Framework
**Author:** Felipe Taha Sant'Ana  
**Scope:** Multi-model benchmarking, operational metrics, hallucination detection, agent trajectory analysis, statistical testing

---
## Overview
A comprehensive evaluation suite for comparing Large Language Models across multiple dimensions. The framework simulates evaluating **6 models** across **8 task categories** with 200 questions each (9,600 total evaluations) plus 500 agent trajectories.

### What This Project Demonstrates
- **Quality benchmarking** — accuracy heatmaps, radar charts, per-task analysis
- **Operational metrics** — latency percentiles (p50/p95/p99), cost analysis
- **Cost-accuracy Pareto frontier** — finding optimal price-quality tradeoffs
- **Hallucination detection** — by model and by task category
- **LLM-as-Judge calibration** — does the judge's confidence match reality?
- **Agent trajectory analysis** — tool usage patterns, error/retry rates, success by complexity
- **Statistical significance testing** — pairwise z-tests for model comparisons

## Contents
1. [Eval Dataset Generation](#1) — 2. [Quality Benchmarks](#2) — 3. [Operational Metrics](#3)
4. [Hallucination & Safety](#4) — 5. [Statistical Testing](#5) — 6. [Agent Evaluation](#6) — 7. [Conclusions](#7)

---
<a id="1"></a>
## 1. Evaluation Dataset Generation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary':'#7C3AED','secondary':'#0EA5E9','accent':'#F59E0B','danger':'#EF4444',
          'success':'#10B981','dark':'#0F172A','rose':'#F43F5E','indigo':'#6366F1'}
palette = list(COLORS.values())
np.random.seed(42)
print("Libraries loaded")

In [ ]:
# ── Define models, tasks, and latent skills ──────────────────────────────
models = ['GPT-class-Large','GPT-class-Mini','Claude-class-Sonnet',
          'Claude-class-Haiku','OSS-70B','OSS-8B']
task_categories = ['Reasoning','Coding','Math','Knowledge_QA',
                   'Summarization','Translation','Tool_Use','Safety']
n_questions_per_task = 200

model_skills = {
    'GPT-class-Large':     {'Reasoning':0.88,'Coding':0.82,'Math':0.85,'Knowledge_QA':0.86,
                             'Summarization':0.91,'Translation':0.89,'Tool_Use':0.84,'Safety':0.92},
    'GPT-class-Mini':      {'Reasoning':0.74,'Coding':0.71,'Math':0.69,'Knowledge_QA':0.78,
                             'Summarization':0.85,'Translation':0.82,'Tool_Use':0.72,'Safety':0.88},
    'Claude-class-Sonnet': {'Reasoning':0.91,'Coding':0.87,'Math':0.83,'Knowledge_QA':0.84,
                             'Summarization':0.93,'Translation':0.88,'Tool_Use':0.89,'Safety':0.95},
    'Claude-class-Haiku':  {'Reasoning':0.78,'Coding':0.75,'Math':0.71,'Knowledge_QA':0.79,
                             'Summarization':0.87,'Translation':0.83,'Tool_Use':0.76,'Safety':0.91},
    'OSS-70B':             {'Reasoning':0.81,'Coding':0.79,'Math':0.76,'Knowledge_QA':0.80,
                             'Summarization':0.83,'Translation':0.78,'Tool_Use':0.74,'Safety':0.82},
    'OSS-8B':              {'Reasoning':0.62,'Coding':0.58,'Math':0.54,'Knowledge_QA':0.68,
                             'Summarization':0.75,'Translation':0.72,'Tool_Use':0.55,'Safety':0.78},
}

# Generate per-question results
records = []
for model in models:
    for task in task_categories:
        true_acc = model_skills[model][task]
        difficulties = np.random.beta(2, 2, n_questions_per_task)
        for i in range(n_questions_per_task):
            p_correct = np.clip(true_acc - 0.3 * difficulties[i] + 0.15, 0.05, 0.98)
            correct = np.random.random() < p_correct
            base_lat = {'GPT-class-Large':2400,'GPT-class-Mini':800,'Claude-class-Sonnet':2100,
                        'Claude-class-Haiku':600,'OSS-70B':3500,'OSS-8B':400}[model]
            latency = np.random.lognormal(np.log(base_lat), 0.25)
            input_tokens = np.random.poisson(150)
            output_tokens = int(np.random.lognormal(5.0, 0.6))
            cost_per_mtok = {'GPT-class-Large':10.0,'GPT-class-Mini':0.30,'Claude-class-Sonnet':15.0,
                             'Claude-class-Haiku':0.80,'OSS-70B':2.0,'OSS-8B':0.20}[model]
            cost = (input_tokens + output_tokens) / 1e6 * cost_per_mtok * 100
            halluc_prob = (1 - true_acc) * 0.4 + difficulties[i] * 0.2
            hallucinated = (not correct) and (np.random.random() < halluc_prob)
            judge_score = np.clip(np.random.normal(true_acc, 0.15) if correct
                                   else np.random.normal(true_acc - 0.4, 0.20), 0, 1)
            records.append({'model':model,'task':task,'difficulty':difficulties[i],
                'correct':int(correct),'latency_ms':latency,'cost_cents':cost,
                'hallucinated':int(hallucinated),'judge_score':judge_score})

df = pd.DataFrame(records)
print(f"Eval records: {len(df):,} = {len(models)} models × {len(task_categories)} tasks × {n_questions_per_task} Q")
df.head()

<a id="2"></a>
## 2. Quality Benchmarks
### Overall Accuracy with 95% Confidence Intervals

In [ ]:
overall = df.groupby('model').agg(accuracy=('correct','mean'), n=('correct','count')).reset_index().sort_values('accuracy')
overall['ci'] = 1.96 * np.sqrt(overall['accuracy']*(1-overall['accuracy'])/overall['n'])

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(overall['model'], overall['accuracy']*100, xerr=overall['ci']*100,
                color=palette[:len(overall)], edgecolor='white', capsize=5)
ax.set_title('Overall Accuracy by Model (95% CI)', fontweight='bold', fontsize=14)
ax.set_xlabel('Accuracy (%)'); ax.set_xlim(0, 100)
for b, v in zip(bars, overall['accuracy']*100):
    ax.text(v+1, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontweight='bold')
plt.tight_layout(); plt.show()

### Performance Heatmap (Model × Task)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
heatmap_data = df.groupby(['model','task'])['correct'].mean().unstack() * 100
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', center=75,
            linewidths=1.5, ax=ax, cbar_kws={'label':'Accuracy (%)'},
            annot_kws={'fontweight':'bold'})
ax.set_title('Model Performance Matrix (% Accuracy)', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

### Radar Charts — Multi-Dimensional Skill Profiles

In [ ]:
fig = plt.figure(figsize=(14, 7))
n_axes = len(task_categories)
angles = np.linspace(0, 2*np.pi, n_axes, endpoint=False).tolist() + [0]
for i, (subset, title) in enumerate([
    (['GPT-class-Large','Claude-class-Sonnet','OSS-70B'], 'Premium Models'),
    (['GPT-class-Mini','Claude-class-Haiku','OSS-8B'], 'Efficient Models')]):
    ax = fig.add_subplot(1, 2, i+1, projection='polar')
    for j, model in enumerate(subset):
        scores = [model_skills[model][t]*100 for t in task_categories] + [model_skills[model][task_categories[0]]*100]
        ax.plot(angles, scores, linewidth=2.5, color=palette[j], label=model)
        ax.fill(angles, scores, alpha=0.15, color=palette[j])
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(task_categories, fontsize=9)
    ax.set_ylim(0, 100); ax.set_title(title, pad=20, fontweight='bold')
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=8)
plt.tight_layout(); plt.show()

<a id="3"></a>
## 3. Operational Metrics
### Latency Distribution

In [ ]:
lat_stats = df.groupby('model')['latency_ms'].agg(['median',
    lambda x: np.percentile(x, 95), lambda x: np.percentile(x, 99)]).reset_index()
lat_stats.columns = ['model','p50','p95','p99']
lat_stats = lat_stats.sort_values('p50')

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
x = np.arange(len(lat_stats)); w = 0.27
axes[0].bar(x-w, lat_stats['p50'], w, color=COLORS['primary'], label='p50', edgecolor='white')
axes[0].bar(x,   lat_stats['p95'], w, color=COLORS['accent'], label='p95', edgecolor='white')
axes[0].bar(x+w, lat_stats['p99'], w, color=COLORS['danger'], label='p99', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(lat_stats['model'], rotation=20, ha='right', fontsize=9)
axes[0].set_title('Latency Percentiles by Model', fontweight='bold')
axes[0].set_ylabel('Latency (ms)'); axes[0].legend()

sns.violinplot(data=df, x='model', y='latency_ms', palette=palette[:6], ax=axes[1], inner='quartile')
axes[1].set_title('Latency Distribution (Log Scale)', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20, ha='right', fontsize=9)
axes[1].set_yscale('log')
plt.tight_layout(); plt.show()

### Cost-Accuracy Pareto Frontier

In [ ]:
cost_stats = df.groupby('model').agg(avg_cost=('cost_cents','mean'),
    accuracy=('correct','mean')).reset_index().sort_values('avg_cost')

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(cost_stats['avg_cost'], cost_stats['accuracy']*100, s=400,
           c=palette[:6], edgecolor='white', linewidth=2, zorder=5)
for _, row in cost_stats.iterrows():
    ax.annotate(row['model'], (row['avg_cost'], row['accuracy']*100),
                xytext=(8, 8), textcoords='offset points', fontsize=10, fontweight='bold')

# Pareto frontier
pareto_pts = []; max_acc = 0
for _, row in cost_stats.iterrows():
    if row['accuracy'] > max_acc:
        pareto_pts.append((row['avg_cost'], row['accuracy']*100))
        max_acc = row['accuracy']
if len(pareto_pts) >= 2:
    px, py = zip(*pareto_pts)
    ax.plot(px, py, '--', color=COLORS['danger'], linewidth=2.5, alpha=0.7, label='Pareto Frontier')
    ax.legend(fontsize=11, loc='lower right')

ax.set_title('Cost-Accuracy Pareto Frontier', fontweight='bold', fontsize=14)
ax.set_xlabel('Avg Cost per Query (cents)'); ax.set_ylabel('Accuracy (%)')
ax.set_xscale('log')
plt.tight_layout(); plt.show()
print("\nModels on the Pareto frontier offer the best quality-cost tradeoff.")

<a id="4"></a>
## 4. Hallucination & Safety Analysis

In [ ]:
halluc_rate = df.groupby('model')['hallucinated'].mean().sort_values() * 100
fig, ax = plt.subplots(figsize=(11, 6))
colors = [COLORS['success'] if v < 5 else COLORS['accent'] if v < 10 else COLORS['danger'] for v in halluc_rate.values]
bars = ax.barh(halluc_rate.index, halluc_rate.values, color=colors, edgecolor='white')
ax.set_title('Hallucination Rate by Model (%)', fontweight='bold', fontsize=14)
ax.set_xlabel('Hallucination Rate (%)')
for b, v in zip(bars, halluc_rate.values):
    ax.text(v+0.2, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontweight='bold')
plt.tight_layout(); plt.show()

### LLM-as-Judge Calibration
Does the judge's confidence track real correctness?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for i, model in enumerate(['GPT-class-Large','Claude-class-Sonnet','OSS-8B']):
    sub = df[df['model'] == model].copy()
    sub['judge_bin'] = pd.cut(sub['judge_score'], bins=10)
    cal = sub.groupby('judge_bin', observed=True).agg(
        mean_judge=('judge_score','mean'),
        actual_correct=('correct','mean')).reset_index()
    ax.plot(cal['mean_judge'], cal['actual_correct'], 'o-', linewidth=2.5,
            markersize=10, color=palette[i], label=model)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect Calibration')
ax.set_title('LLM-as-Judge Calibration', fontweight='bold', fontsize=14)
ax.set_xlabel('Judge Score'); ax.set_ylabel('Actual Accuracy'); ax.legend()
plt.tight_layout(); plt.show()
print("Well-calibrated judges sit close to the diagonal.")

<a id="5"></a>
## 5. Statistical Significance Testing
### Pairwise Two-Proportion z-Tests

In [ ]:
# Compute pairwise p-values and accuracy differences
pairwise_pvals = pd.DataFrame(index=models, columns=models, dtype=float)
pairwise_diffs = pd.DataFrame(index=models, columns=models, dtype=float)
for m1 in models:
    for m2 in models:
        if m1 == m2:
            pairwise_pvals.loc[m1,m2] = 1.0; pairwise_diffs.loc[m1,m2] = 0.0; continue
        s1, s2 = df[df['model']==m1]['correct'], df[df['model']==m2]['correct']
        p1, p2 = s1.mean(), s2.mean(); n1, n2 = len(s1), len(s2)
        p_pool = (s1.sum() + s2.sum()) / (n1 + n2)
        se = np.sqrt(p_pool * (1-p_pool) * (1/n1 + 1/n2))
        z = (p1 - p2) / se
        pairwise_pvals.loc[m1,m2] = 2 * (1 - stats.norm.cdf(abs(z)))
        pairwise_diffs.loc[m1,m2] = (p1 - p2) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(pairwise_diffs.astype(float), annot=True, fmt='+.1f', cmap='RdBu_r', center=0,
            ax=axes[0], linewidths=1, annot_kws={'fontweight':'bold','fontsize':9})
axes[0].set_title('Accuracy Difference (Row − Column, pp)', fontweight='bold')
axes[0].set_xticklabels(models, rotation=30, ha='right', fontsize=9)

pvals_arr = pairwise_pvals.astype(float).values
sig_labels = np.where(pvals_arr < 0.001, '***',
              np.where(pvals_arr < 0.01, '**',
              np.where(pvals_arr < 0.05, '*', 'n.s.')))
np.fill_diagonal(sig_labels, '—')
sns.heatmap(-np.log10(pvals_arr.clip(min=1e-50)), annot=sig_labels, fmt='', cmap='YlOrRd',
            xticklabels=models, yticklabels=models, ax=axes[1], linewidths=1,
            annot_kws={'fontweight':'bold','fontsize':9})
axes[1].set_title('Pairwise Significance', fontweight='bold')
axes[1].set_xticklabels(models, rotation=30, ha='right', fontsize=9)
plt.tight_layout(); plt.show()
print("\n*** = p<0.001, ** = p<0.01, * = p<0.05, n.s. = not significant")

<a id="6"></a>
## 6. Agent Trajectory Analysis

Beyond simple Q&A, agents must plan, use tools, and recover from errors. We analyze 500 multi-step trajectories.

In [ ]:
# Generate agent trajectory data
n_trajectories = 500
trajectories = []
tools_used = ['search', 'calculator', 'code_exec', 'file_read', 'api_call']
for i in range(n_trajectories):
    model = np.random.choice(models)
    task_complexity = np.random.choice(['Simple','Medium','Complex'], p=[0.40, 0.40, 0.20])
    expected_steps = {'Simple':3,'Medium':7,'Complex':14}[task_complexity]
    actual_steps = max(1, int(np.random.normal(expected_steps, expected_steps*0.3)))
    tool_calls = {t: np.random.poisson(0.4 * actual_steps / len(tools_used)) for t in tools_used}
    n_errors = np.random.poisson(0.15 * actual_steps)
    n_retries = max(0, int(np.random.normal(n_errors*0.6, 1)))
    skill = np.mean([model_skills[model]['Reasoning'], model_skills[model]['Tool_Use']])
    p_success = skill - {'Simple':0,'Medium':0.10,'Complex':0.25}[task_complexity]
    success = np.random.random() < p_success
    trajectories.append({'model':model,'task_complexity':task_complexity,
        'expected_steps':expected_steps,'actual_steps':actual_steps,
        'success':int(success),'n_errors':n_errors,'n_retries':n_retries,
        **{f'tool_{t}':tool_calls[t] for t in tools_used}})
df_traj = pd.DataFrame(trajectories)
print(f"Trajectories: {len(df_traj):,}")
df_traj.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
success_by = df_traj.groupby(['model','task_complexity'])['success'].mean().unstack() * 100
success_by = success_by[['Simple','Medium','Complex']]
success_by.plot(kind='bar', ax=axes[0],
    color=[COLORS['success'],COLORS['accent'],COLORS['danger']], edgecolor='white')
axes[0].set_title('Agent Success Rate by Task Complexity', fontweight='bold')
axes[0].set_ylabel('Success Rate (%)'); axes[0].legend(title='Complexity')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20, ha='right', fontsize=9)
axes[0].set_ylim(0, 105)

err_stats = df_traj.groupby('model').agg(avg_errors=('n_errors','mean'),
    avg_retries=('n_retries','mean'),
    success_rate=('success','mean')).reset_index().sort_values('success_rate', ascending=False)
x = np.arange(len(err_stats)); w = 0.4
axes[1].bar(x-w/2, err_stats['avg_errors'], w, color=COLORS['danger'], label='Avg Errors', edgecolor='white')
axes[1].bar(x+w/2, err_stats['avg_retries'], w, color=COLORS['accent'], label='Avg Retries', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(err_stats['model'], rotation=20, ha='right', fontsize=9)
axes[1].set_title('Errors & Retries per Trajectory', fontweight='bold')
axes[1].set_ylabel('Count'); axes[1].legend()
plt.tight_layout(); plt.show()

<a id="7"></a>
## 7. Conclusions

### Final Leaderboard
| Model | Accuracy | p50 Latency | Avg Cost | Hallucination |
|:------|:--------:|:-----------:|:--------:|:-------------:|
| Claude-class-Sonnet | **88.8%** | 2,096 ms | 0.481¢ | Low |
| GPT-class-Large | 87.9% | 2,384 ms | 0.327¢ | Low |
| OSS-70B | 80.2% | 3,499 ms | 0.065¢ | Medium |
| Claude-class-Haiku | 79.2% | 601 ms | 0.026¢ | Low |
| GPT-class-Mini | 78.9% | 797 ms | 0.010¢ | Low-Medium |
| OSS-8B | 67.4% | 403 ms | 0.007¢ | Higher |

### Recommendations by Use Case
- **Best overall quality** → Claude-class-Sonnet (highest accuracy + safety)
- **Best latency** → OSS-8B (~400ms p50)
- **Best cost-efficiency** → OSS-8B (0.007¢/query)
- **Best balance** → GPT-class-Mini (78% accuracy at 0.010¢)
- **Best for agents** → Claude-class-Sonnet (highest tool-use score)

### Framework Capabilities
✅ Multi-model benchmarking across 8 task categories  
✅ Operational metrics (latency p50/p95/p99, cost)  
✅ Cost-accuracy Pareto analysis  
✅ Hallucination & safety scoring  
✅ LLM-as-Judge calibration  
✅ Pairwise statistical significance testing  
✅ Agent trajectory analysis (tools, errors, retries)  

### Future Work
- **Human eval integration** (Likert ratings, preference comparisons)
- **Adversarial / red-teaming** test suite
- **Context length scaling** experiments
- **Multi-turn conversation** evaluation
- **RAG-specific metrics** (faithfulness, context relevance)
- **Public benchmark integration** (MMLU, HumanEval, GPQA, SWE-bench)
- **CI/CD integration** for regression testing on every model release

---
*Synthetic data simulates a realistic LLM evaluation scenario. Model names are illustrative.*
